# 헬스케어 보험료 예측 회귀모델 구현

보험료를 예측하는 모델을 만들어라.

In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('insurance.csv')

print(f"데이터 크기 (행, 열): {df.shape}")
print(df.head())

데이터 크기 (행, 열): (1338, 7)
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [5]:
df['smoker'] = df['smoker'].astype(str).str.strip().str.lower()
df['sex'] = df['sex'].astype(str).str.strip().str.lower()
df['region'] = df['region'].astype(str).str.strip().str.lower()

# 범주형 변수 숫자로 변환 (원핫 인코딩 적용)
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True, dtype=int)

print("[원핫 인코딩 후 데이터 컬럼 구조]")
print(df_encoded.head())

# 독립변수(X)와 종속변수(y) 분리
# 예측할 대상인 보험료(charges)를 y로, 나머지를 예측에 사용할 변수(X)로 분리
X = df_encoded.drop(columns=['charges', 'age_group', 'age_group_name'], errors='ignore')
y = df_encoded['charges']

print("[독립변수(X) 목록]")
print(X.columns.tolist())
print(f"\nX의 크기: {X.shape}, y의 크기: {y.shape}")

[원핫 인코딩 후 데이터 컬럼 구조]
   age     bmi  children      charges  sex_male  smoker_yes  region_northwest  \
0   19  27.900         0  16884.92400         0           1                 0   
1   18  33.770         1   1725.55230         1           0                 0   
2   28  33.000         3   4449.46200         1           0                 0   
3   33  22.705         0  21984.47061         1           0                 1   
4   32  28.880         0   3866.85520         1           0                 1   

   region_southeast  region_southwest  
0                 0                 1  
1                 1                 0  
2                 1                 0  
3                 0                 0  
4                 0                 0  
[독립변수(X) 목록]
['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']

X의 크기: (1338, 8), y의 크기: (1338,)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

In [6]:
# X(나이, BMI, 자녀수, 흡연여부)와 y(보험료)를 8:2 비율로 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"학습용 독립변수 (X_train) 크기: {X_train.shape}")
print(f"검증용 독립변수 (X_test) 크기: {X_test.shape}")

학습용 독립변수 (X_train) 크기: (1070, 8)
검증용 독립변수 (X_test) 크기: (268, 8)


In [11]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=0.01),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5)
}

# 각 모델을 학습
for name, model in models.items():
    model.fit(X_train, y_train)

print("=== 4종 모델 학습 완료 ===")

=== 4종 모델 학습 완료 ===


In [12]:
results_list = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    
    # 평가지표
    train_score = model.score(X_train, y_train)   # 학습 데이터 R2 Score
    test_score = model.score(X_test, y_test)      # 검증 데이터 R2 Score
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    # 결과를 리스트에 저장
    results_list.append({
        'Model': name,
        'Train R2 Score': round(train_score, 4),
        'Test R2 Score': round(test_score, 4),
        'MSE': round(mse, 2),
        'RMSE': round(rmse, 2)
    })

# 한눈에 비교하기 위해 데이터프레임으로 변환하여 출력
results_df = pd.DataFrame(results_list)
display(results_df)

,Model,Train R2 Score,Test R2 Score,MSE,RMSE
0,Linear Regression,0.7417,0.7836,33596915.85,5796.28
1,Ridge,0.7417,0.7836,33597370.84,5796.32
2,Lasso,0.7417,0.7836,33597001.22,5796.29
3,ElasticNet,0.7417,0.7834,33622010.33,5798.45


* 위 결과에 따라, 현 보험목록에서는 선형회귀모델이 가장 성능이 좋은 모델로 확인됨
  - 변수 종류가 많지 않아 과적합 문제가 많이 없고, 애초에 클린한 데이터였기 때문에

In [13]:
import joblib

final_model = models['Linear Regression']
joblib.dump(final_model, 'insurance_premium_predict_model.pkl')
print("=== 최종 모델 저장 완료 (insurance_premium_predict_model.pkl) ===")

=== 최종 모델 저장 완료 (insurance_premium_predict_model.pkl) ===


In [14]:
loaded_model = joblib.load('insurance_premium_predict_model.pkl')

In [16]:
new_customers = pd.DataFrame([
    # 고객 A: 25세, BMI 22.5, 자녀 0명, 여성(0), 비흡연(0), northeast 거주
    [25, 22.5, 0, 0, 0, 0, 0, 0],  
    
    # 고객 B: 25세, BMI 33.2, 자녀 0명, 남성(1), 흡연자(1), southeast 거주
    [25, 33.2, 0, 1, 1, 0, 1, 0],  
    
    # 고객 C: 55세, BMI 28.0, 자녀 2명, 여성(0), 비흡연(0), southwest 거주
    [55, 28.0, 2, 0, 0, 0, 0, 1]   
], columns=['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest'])


predicted_charges = loaded_model.predict(new_customers)

print("=== 새로운 고객별 예상 보험료 예측 결과 ===")
for i, charge in enumerate(predicted_charges):
    print(f"고객 {chr(65+i)}의 예상 연간 보험료: ${charge:,.2f} (약 {int(charge * 1350):,}원)")

=== 새로운 고객별 예상 보험료 예측 결과 ===
고객 A의 예상 연간 보험료: $2,077.76 (약 2,804,970원)
고객 B의 예상 연간 보험료: $28,659.32 (약 38,690,080원)
고객 C의 예상 연간 보험료: $11,681.79 (약 15,770,422원)


In [ ]:
# 학습된 모델에서 가중치(기울기)와 y절편(기본 상수) 추출
coefficients = final_model.coef_
intercept = final_model.intercept_

# 가중치를 데이터프레임으로 정리
feature_names = X.columns
weights_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight (가중치/달러)': np.round(coefficients, 2)
})

print("=== 기존 데이터에 근거하여 모델이 학습한 내부 가중치 ===")
# 0세, BMI 0, 자녀 0명, 비흡연자, 여성 기준  # 18세, BMI 16 등 최솟값 대입 시 플러스 숫자로 전환
print(f"기본 베이스 보험료 (y절편): ${intercept:,.2f}")   
display(weights_df)

=== 기존 데이터에 근거하여 모델이 학습한 내부 가중치 ===
기본 베이스 보험료 (y절편): $-11,931.22


,Feature,Weight (가중치/달러)
0,age,256.98
1,bmi,337.09
2,children,425.28
3,sex_male,-18.59
4,smoker_yes,23651.13
5,region_northwest,-370.68
6,region_southeast,-657.86
7,region_southwest,-809.80


In [19]:
# 고객 A의 데이터를 바탕으로 수학적 수식 검증
# 고객 A 정보: 25세, BMI 22.5, 자녀 0명, 여성(0), 비흡연(0), northeast 거주
print("=== 고객 A의 실제 수식 검증 ===")
calc_premium = (
    intercept 
    + (coefficients[0] * 25)  # 나이(age) 효과
    + (coefficients[1] * 22.5) # BMI 효과
    + (coefficients[2] * 0)    # 자녀수 효과
    + (coefficients[3] * 0)    # 성별 효과
    + (coefficients[4] * 0)    # 흡연 효과
    + (coefficients[5] * 0)    # 지역 효과
    + (coefficients[6] * 0)
    + (coefficients[7] * 0)
)
print(f"수식으로 직접 계산한 보험료 : ${calc_premium:,.2f}")
print(f"모델이 예측했던 고객 A 보험료: ${predicted_charges[0]:,.2f}")

=== 고객 A의 실제 수식 검증 ===
수식으로 직접 계산한 보험료 : $2,077.76
모델이 예측했던 고객 A 보험료: $2,077.76
